# Phase 2B — Probes & Ablations: *what did the classifier actually learn?*
Requires Phase 1 (`processed/`) **and** Phase 2A checkpoints (`processed/results2A/`). No retraining
of the 2A grid — everything here reuses those weights, plus two small new trainings (B4, B5).

| # | Experiment | Question | Cost |
|---|---|---|---|
| B1 | Linear probes on frozen R2 features | Does the representation encode the **singer (126-way)** better than the **genre (15-way)**? Plus gender + source-domain (production signature). | ~10 min |
| B2 | Time-shuffle + frequency-band masking | Is it melody/rhythm or a **bag of spectra**? Voice band or percussion/mastering band? | ~10–35 min |
| B3 | MFCC-stats + RandomForest baseline | Does deep learning add anything beyond **average timbre** under each regime? | ~20–30 min (CPU) |
| B4 | Shuffled-label control | Pipeline sanity: memorization ⇒ chance on song-disjoint test. | ~3–12 min |
| B5 | **Artist→genre oracle** | Train a pure 126-way *singer* classifier, route through the artist→genre table. If it matches the direct genre model, the shortcut is proven end-to-end. | ~20–90 min |
| B6 | Artist-count scatter + confusion matrices | Mechanism check (Tamang Selo) + where does **Bauls** go? | ~2 min |

All outputs → `$IKS_RES2B` (default `processed/results2B/`). B5 is resumable per fold.


In [ ]:
# ============ CELL 0: smoke-test switch ============
import os
os.environ['IKS_SMOKE'] = '0'   # '1' = tiny quick pass to verify the pipeline; '0' = full

In [ ]:
# ============ CELL 1: Setup (same data layer as 2A) ============
import os, gc, glob, json, time, shutil, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

if os.path.basename(os.getcwd()) == 'notebooks': os.chdir('..')   # resolve paths from the repo root
PROC_ROOT  = os.environ.get('IKS_PROC_ROOT', './processed')
RES2A      = os.environ.get('IKS_RES2A', f"{PROC_ROOT}/results2A")
RES2B      = os.environ.get('IKS_RES2B', f"{PROC_ROOT}/results2B")
os.makedirs(RES2B, exist_ok=True)
SMOKE = os.environ.get('IKS_SMOKE', '0') == '1'
assert glob.glob(f"{RES2A}/R2_resnet18_fold*.pt"), "2A checkpoints not found — run Phase 2A first"

idxdf = pd.read_parquet(f"{PROC_ROOT}/index_with_folds.parquet").sort_values('global_idx').reset_index(drop=True)
assert (idxdf['global_idx'].values == np.arange(len(idxdf))).all()
idxdf = idxdf.set_index('global_idx', drop=False)
for c in ['genre_id','artist_id','state_id','gender_id','fold_artist','fold_song']:
    idxdf[c] = idxdf[c].astype(int)
N_TOTAL = len(idxdf); N_CLASSES = idxdf['genre_id'].nunique()
ID2GENRE = idxdf.drop_duplicates('genre_id').set_index('genre_id')['genre'].to_dict()

blocks = idxdf.groupby('style')['global_idx'].agg(['min','max','count']).sort_values('min')
X_ALL = np.empty((N_TOTAL, 128, 130), dtype=np.float16)
t0 = time.time()
for s, r in blocks.iterrows():
    Xs = np.load(f"{PROC_ROOT}/X_{s}.npy", mmap_mode='r'); X_ALL[r['min']:r['max']+1] = Xs[:]; del Xs
print(f"X_ALL in RAM: {X_ALL.nbytes/1e9:.2f} GB ({time.time()-t0:.0f}s)")

STRIDE = 6
idxdf['seg_rank'] = idxdf.groupby('source').cumcount()
stride_pool = idxdf.loc[idxdf['seg_rank'] % STRIDE == 0, 'global_idx'].values
full_pool   = idxdf['global_idx'].values

# label arrays indexed by global_idx (row i == global_idx i)
GENRE_ARR  = idxdf['genre_id'].values
ART_UNIQ   = np.sort(idxdf['artist_id'].unique())
ART2IDX    = {a: i for i, a in enumerate(ART_UNIQ)}
ARTIST_ARR = idxdf['artist_id'].map(ART2IDX).values.astype(np.int64)
N_ARTISTS  = len(ART_UNIQ)
GENDER_ARR = idxdf['gender_id'].values

def domain_of(u):
    u = str(u).lower()
    if 'gaana'  in u: return 0
    if 'youtu'  in u: return 1
    return 2
idxdf['domain_id'] = idxdf['source'].map(domain_of)
DOMAIN_ARR = idxdf['domain_id'].values
dom_counts = idxdf.drop_duplicates('source')['domain_id'].value_counts().rename({0:'gaana',1:'youtube',2:'other'})
print("recordings by source domain:", dom_counts.to_dict())
print(f"classes: genre={N_CLASSES}  artist={N_ARTISTS}  | stride pool {len(stride_pool):,} | SMOKE={SMOKE}")

# artist -> majority genre lookup (cross-style artists get their majority genre; noted in paper)
rec = idxdf.drop_duplicates('source')
ART_MAJ_GENRE = rec.groupby('artist_id')['genre_id'].agg(lambda s: s.mode().iloc[0])
ART_IDX_2_GENRE = np.array([ART_MAJ_GENRE[a] for a in ART_UNIQ], dtype=np.int64)

import torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else '(CPU)')

In [ ]:
# ============ CELL 2: Models (verbatim 2A), loaders, transforms, generic trainer ============
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import f1_score, accuracy_score

def normalize_batch(x): return (x + 40.0) / 40.0

class BasicBlock(nn.Module):
    def __init__(self, cin, cout, stride=1):
        super().__init__()
        self.c1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False); self.b1 = nn.BatchNorm2d(cout)
        self.c2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False);     self.b2 = nn.BatchNorm2d(cout)
        self.down = None
        if stride != 1 or cin != cout:
            self.down = nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False), nn.BatchNorm2d(cout))
    def forward(self, x):
        idn = x if self.down is None else self.down(x)
        h = F.relu(self.b1(self.c1(x))); h = self.b2(self.c2(h))
        return F.relu(h + idn)

class ResNet18(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(1, 64, 7, 2, 3, bias=False), nn.BatchNorm2d(64),
                                  nn.ReLU(inplace=True), nn.MaxPool2d(3, 2, 1))
        def layer(cin, cout, s): return nn.Sequential(BasicBlock(cin, cout, s), BasicBlock(cout, cout))
        self.l1 = layer(64, 64, 1);  self.l2 = layer(64, 128, 2)
        self.l3 = layer(128, 256, 2); self.l4 = layer(256, 512, 2)
        self.fc = nn.Linear(512, n_classes)
    def embed(self, x):
        h = self.stem(x); h = self.l4(self.l3(self.l2(self.l1(h))))
        return F.adaptive_avg_pool2d(h, 1).flatten(1)
    def forward(self, x): return self.fc(self.embed(x))

class CRNN(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        def blk(cin, cout): return nn.Sequential(nn.Conv2d(cin, cout, 3, 1, 1), nn.BatchNorm2d(cout),
                                                 nn.ReLU(inplace=True), nn.MaxPool2d(2))
        self.conv = nn.Sequential(blk(1, 32), blk(32, 64), blk(64, 128), blk(128, 128))
        self.gru = nn.GRU(128 * 8, 128, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(256, n_classes)
    def embed(self, x):
        h = self.conv(x); h = h.permute(0, 3, 1, 2).flatten(2)
        out, _ = self.gru(h); return out.mean(1)
    def forward(self, x): return self.fc(self.embed(x))

def make_model(name, n_classes): return {'resnet18': ResNet18, 'crnn': CRNN}[name](n_classes)

def load_ckpt(name, path, n_classes=None):
    m = make_model(name, n_classes or N_CLASSES).to(DEVICE)
    m.load_state_dict(torch.load(path, map_location=DEVICE)); m.eval(); return m

def metric_pack(y, p, prefix):
    labels = np.unique(y)
    return {f'{prefix}_acc': float(accuracy_score(y, p)),
            f'{prefix}_macro_f1': float(f1_score(y, p, labels=labels, average='macro', zero_division=0))}

def song_level(gidx, logits, label_arr=GENRE_ARR):
    src = idxdf.loc[gidx, 'source'].values
    L = pd.DataFrame(logits); L['source'] = src
    agg = L.groupby('source').mean()
    ytrue = pd.Series(label_arr[gidx], index=src).groupby(level=0).first().loc[agg.index].values
    return ytrue, agg.values.argmax(1)

@torch.no_grad()
def predict(model, gidx, transform=None, batch=1024, use_embed=False):
    model.eval(); outs = []
    for i in range(0, len(gidx), batch):
        xb = X_ALL[gidx[i:i+batch]].astype(np.float32)
        if transform is not None: xb = transform(xb)
        xb = normalize_batch(torch.from_numpy(xb).unsqueeze(1)).to(DEVICE, non_blocking=True)
        with torch.autocast(DEVICE.type, enabled=(DEVICE.type == 'cuda')):
            out = model.embed(xb) if use_embed else model(xb)
        outs.append(out.float().cpu())
    return torch.cat(outs).numpy()

# ---- input-space transforms (applied to raw dB before normalization) ----
def make_shuffle_time(seed):
    rng = np.random.default_rng(seed)
    def f(x):                                   # same frame permutation across all mel bins of a segment
        for i in range(x.shape[0]): x[i] = x[i][:, rng.permutation(x.shape[2])]
        return x
    return f

def make_mask(lo, hi):
    def f(x): x[:, lo:hi, :] = -80.0; return x
    return f

def mel_bin_hz(b, n_mels=128, fmax=11025.0):    # approx center frequency of mel bin b
    m_max = 2595.0 * np.log10(1 + fmax / 700.0)
    m = (b + 0.5) / n_mels * m_max
    return 700.0 * (10 ** (m / 2595.0) - 1)

def build_indices(regime, fold):                # identical to 2A
    if regime in ('R1a', 'R1b'):
        pool = full_pool if regime == 'R1a' else stride_pool
        perm = np.random.default_rng(100).permutation(pool)
        chunks = np.array_split(perm, 5)
        te = chunks[fold]; tr = np.concatenate([c for j, c in enumerate(chunks) if j != fold])
    else:
        col = 'fold_song' if regime == 'R2' else 'fold_artist'
        pool = stride_pool; f = idxdf.loc[pool, col].values
        te, tr = pool[f == fold], pool[f != fold]
    return tr, te

def carve_val_by(tr, key, fold, frac=0.10):
    rng = np.random.default_rng(1000 + fold)
    if key is None:
        perm = rng.permutation(tr); nv = int(len(tr) * frac); return perm[nv:], perm[:nv]
    g = idxdf.loc[tr, key].values
    ug = rng.permutation(np.unique(g)); nv = max(1, int(round(len(ug) * frac)))
    vmask = np.isin(g, ug[:nv]); return tr[~vmask], tr[vmask]

def train_generic(model_name, tr, va, te, label_arr, n_classes, epochs, batch=512, lr=3e-4, seed=0, log_every=10):
    torch.manual_seed(seed); np.random.seed(seed)
    model = make_model(model_name, n_classes).to(DEVICE)
    counts = np.bincount(label_arr[tr], minlength=n_classes).astype(np.float64)
    w = np.where(counts > 0, counts.sum() / (n_classes * np.maximum(counts, 1)), 0.0)
    crit  = nn.CrossEntropyLoss(weight=torch.tensor(w, dtype=torch.float32, device=DEVICE))
    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    scaler = torch.amp.GradScaler(enabled=(DEVICE.type == 'cuda'))
    rng = np.random.default_rng(seed)
    best_f1, best_state = -1.0, None
    t0 = time.time()
    for ep in range(epochs):
        model.train(); perm = rng.permutation(tr)
        for i in range(0, len(perm), batch):
            gb = perm[i:i+batch]
            xb = normalize_batch(torch.from_numpy(X_ALL[gb].astype(np.float32)).unsqueeze(1)).to(DEVICE)
            yb = torch.from_numpy(label_arr[gb]).to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(DEVICE.type, enabled=(DEVICE.type == 'cuda')):
                loss = crit(model(xb), yb)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        sched.step()
        pv = predict(model, va).argmax(1); yv = label_arr[va]
        f1v = f1_score(yv, pv, labels=np.unique(yv), average='macro', zero_division=0)
        if f1v > best_f1:
            best_f1 = float(f1v)
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        if ep % log_every == 0 or ep == epochs - 1:
            print(f"    ep {ep:>3}  val macroF1={f1v:.4f} ({time.time()-t0:.0f}s)", flush=True)
    model.load_state_dict(best_state)
    logits = predict(model, te)
    return model, logits, best_f1
print('cell 2 ready')

In [ ]:
# ============ CELL 3 (B1): Linear probes on frozen R2 features ============
# Protocol: for each R2 fold, embed the whole stride pool with that fold's frozen ResNet-18,
# fit a linear probe on the TRAIN songs, test on the held-out songs (song-disjoint, artists shared —
# the correct design for an artist probe). Identical protocol for every target, so the numbers are
# directly comparable: genre (15-way) vs artist (126-way) vs gender (2) vs source-domain.

PROBE_TARGETS = {'genre': (GENRE_ARR, N_CLASSES), 'artist': (ARTIST_ARR, N_ARTISTS),
                 'gender': (GENDER_ARR, 2), 'domain': (DOMAIN_ARR, 3)}
PROBE_EPOCHS = 2 if SMOKE else 40

def linear_probe(Etr, ytr, Ete, yte, n_classes, epochs=PROBE_EPOCHS, seed=0):
    torch.manual_seed(seed)
    mu, sd = Etr.mean(0, keepdims=True), Etr.std(0, keepdims=True) + 1e-6
    Xtr = torch.from_numpy((Etr - mu) / sd).to(DEVICE)
    Xte = torch.from_numpy((Ete - mu) / sd).to(DEVICE)
    Ytr = torch.from_numpy(ytr).to(DEVICE)
    counts = np.bincount(ytr, minlength=n_classes).astype(np.float64)
    w = np.where(counts > 0, counts.sum() / (n_classes * np.maximum(counts, 1)), 0.0)
    crit = nn.CrossEntropyLoss(weight=torch.tensor(w, dtype=torch.float32, device=DEVICE))
    clf = nn.Linear(Etr.shape[1], n_classes).to(DEVICE)
    opt = torch.optim.Adam(clf.parameters(), lr=1e-2, weight_decay=1e-4)
    for ep in range(epochs):
        perm = torch.randperm(len(Xtr), device=DEVICE)
        for i in range(0, len(perm), 8192):
            b = perm[i:i+8192]
            opt.zero_grad(set_to_none=True); loss = crit(clf(Xtr[b]), Ytr[b]); loss.backward(); opt.step()
    with torch.no_grad():
        p = clf(Xte).argmax(1).cpu().numpy()
    return p

probe_rows = []
ckpts = sorted(glob.glob(f"{RES2A}/R2_resnet18_fold*.pt"))
print(f"{len(ckpts)} R2 resnet checkpoints found")
for ck in ckpts:
    fold = int(ck.split('fold')[1].split('.')[0])
    model = load_ckpt('resnet18', ck)
    pool = stride_pool if not SMOKE else stride_pool[:4000]
    E = predict(model, pool, use_embed=True)                     # (n, 512)
    fmask = idxdf.loc[pool, 'fold_song'].values == fold
    Etr, Ete, gtr, gte = E[~fmask], E[fmask], pool[~fmask], pool[fmask]
    for tgt, (arr, C) in PROBE_TARGETS.items():
        ytr, yte = arr[gtr], arr[gte]
        if len(np.unique(ytr)) < 2:
            print(f"  fold{fold} {tgt}: <2 classes in train — skipped"); continue
        cover = np.isin(yte, np.unique(ytr))                     # test rows whose label was seen in train
        if cover.sum() == 0:
            print(f"  fold{fold} {tgt}: no test label seen in train — skipped"); continue
        p = linear_probe(Etr, ytr, Ete[cover], yte[cover], C, seed=fold)
        m = metric_pack(yte[cover], p, 'probe')
        maj = np.bincount(ytr, minlength=C).argmax()
        probe_rows.append(dict(fold=fold, target=tgt, n_classes=C,
                               acc=m['probe_acc'], macro_f1=m['probe_macro_f1'],
                               chance=1.0/len(np.unique(ytr)),
                               majority=float((yte[cover] == maj).mean()),
                               coverage=float(cover.mean())))
        print(f"  fold{fold} {tgt:<7} ({C:>3}-way)  acc={m['probe_acc']:.4f}  "
              f"F1={m['probe_macro_f1']:.4f}  majority={probe_rows[-1]['majority']:.3f}", flush=True)
    del model, E; gc.collect()
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()

P = pd.DataFrame(probe_rows)
summ = P.groupby('target').agg(n_classes=('n_classes','first'), acc_mean=('acc','mean'), acc_std=('acc','std'),
                               f1_mean=('macro_f1','mean'), chance=('chance','mean'),
                               majority=('majority','mean'))
print('\n================ PROBE SUMMARY (song-disjoint, frozen R2 features) ================')
print(summ.round(4).to_string())
P.to_csv(f"{RES2B}/probes_R2_resnet18.csv", index=False); summ.to_csv(f"{RES2B}/probes_summary.csv")
print(f"\nHEADLINE: artist ({N_ARTISTS}-way) vs genre ({N_CLASSES}-way) — if artist ≥ genre despite ~8x")
print("more classes, the representation is a singer identifier wearing a genre costume.")

In [ ]:
# ============ CELL 4 (B2): Time-shuffle + frequency-band masking ============
# Clean reference = the saved 2A test metrics. Perturbations run on the same test sets.
LOW, MID, HIGH = (0, 40), (40, 80), (80, 128)
print(f"band edges: LOW < {mel_bin_hz(40):.0f} Hz | MID {mel_bin_hz(40):.0f}-{mel_bin_hz(80):.0f} Hz "
      f"| HIGH > {mel_bin_hz(80):.0f} Hz")

CONDS = {'shuffle_time': None, 'mask_low': make_mask(*LOW), 'mask_mid': make_mask(*MID), 'mask_high': make_mask(*HIGH)}
GRID  = [('R2', 'resnet18'), ('R2', 'crnn'), ('R3', 'resnet18'), ('R3', 'crnn')]
pert_path = f"{RES2B}/perturbations.json"
pert = json.load(open(pert_path)) if os.path.exists(pert_path) else {}

for regime, mname in GRID:
    for ck in sorted(glob.glob(f"{RES2A}/{regime}_{mname}_fold*.pt")):
        fold = int(ck.split('fold')[1].split('.')[0])
        clean = json.load(open(f"{RES2A}/{regime}_{mname}_fold{fold}.json"))
        _, te = build_indices(regime, fold)
        if SMOKE: te = te[:1200]
        model = None
        for cond, tf in CONDS.items():
            if cond.startswith('mask') and mname == 'crnn': continue      # masks: resnet only (cheaper)
            key = f"{regime}_{mname}_f{fold}_{cond}"
            if key in pert: continue
            if model is None: model = load_ckpt(mname, ck)
            transform = make_shuffle_time(fold * 10 + 7) if cond == 'shuffle_time' else tf
            logits = predict(model, te, transform=transform)
            m = metric_pack(GENRE_ARR[te], logits.argmax(1), 'seg')
            pert[key] = dict(regime=regime, model=mname, fold=fold, cond=cond,
                             seg_acc=m['seg_acc'], clean_acc=clean['seg_acc'])
            json.dump(pert, open(pert_path, 'w'))
            print(f"  {key:<38} acc={m['seg_acc']:.4f}  (clean {clean['seg_acc']:.4f})", flush=True)
        if model is not None:
            del model; gc.collect()
            if DEVICE.type == 'cuda': torch.cuda.empty_cache()

PT = pd.DataFrame(pert.values())
piv = (PT.groupby(['regime', 'model', 'cond'])
         .agg(acc=('seg_acc', 'mean'), clean=('clean_acc', 'mean'))
         .assign(drop=lambda t: t['clean'] - t['acc']))
print('\n================ PERTURBATION SUMMARY (mean over folds) ================')
print(piv.round(4).to_string())
piv.to_csv(f"{RES2B}/perturbations_summary.csv")
print("\nRead it as: small shuffle-drop = bag-of-spectra (no temporal/melodic learning);")
print("mask_low vs mask_high tells you voice-band vs percussion/production-band reliance.")

In [ ]:
# ============ CELL 5 (B3): MFCC-stats + RandomForest under R1b / R2 / R3 ============
from scipy.fftpack import dct
from sklearn.ensemble import RandomForestClassifier

def mfcc_feats(gidx, batch=8192):
    out = np.empty((len(gidx), 78), np.float32)
    for i in range(0, len(gidx), batch):
        x = X_ALL[gidx[i:i+batch]].astype(np.float32)              # (B,128,130) dB
        M  = dct(x, type=2, axis=1, norm='ortho')[:, :13, :]       # 13 MFCCs per frame
        d1 = np.diff(M, axis=2); d2 = np.diff(d1, axis=2)
        out[i:i+batch] = np.concatenate([M.mean(2), M.std(2), d1.mean(2), d1.std(2),
                                         d2.mean(2), d2.std(2)], axis=1)
    return out

mrows, cache = [], {}
for regime in ['R1b', 'R2', 'R3']:
    for fold in range(5):
        tr, te = build_indices(regime, fold)
        if SMOKE: tr, te = tr[:3000], te[:1000]
        Ftr = mfcc_feats(tr); Fte = mfcc_feats(te)
        rf = RandomForestClassifier(n_estimators=60 if SMOKE else 300, n_jobs=-1,
                                    class_weight='balanced_subsample', random_state=fold)
        rf.fit(Ftr, GENRE_ARR[tr])
        proba = rf.predict_proba(Fte)
        m = metric_pack(GENRE_ARR[te], proba.argmax(1), 'seg')
        ys, ps = song_level(te, proba)
        m.update(metric_pack(ys, ps, 'song'))
        mrows.append(dict(regime=regime, fold=fold, **m))
        print(f"  MFCC+RF {regime} fold{fold}: seg acc={m['seg_acc']:.4f}  song acc={m['song_acc']:.4f}", flush=True)
        if SMOKE: break

MF = pd.DataFrame(mrows)
summ = MF.groupby('regime').agg(seg_acc=('seg_acc','mean'), seg_std=('seg_acc','std'),
                                song_acc=('song_acc','mean'))
print('\n================ MFCC + RANDOM FOREST (78-dim timbre summary) ================')
print(summ.round(4).to_string())
MF.to_csv(f"{RES2B}/mfcc_rf.csv", index=False)
print("\nCompare with 2A deep results: if this 78-dim timbre baseline lands near ResNet under R1b,")
print("the deep model adds little beyond average timbre when leakage is available.")

In [ ]:
# ============ CELL 6 (B4): Shuffled-label control ============
# Genre labels permuted AT THE RECORDING LEVEL (each source gets a random genre), R2 fold 0.
# Expect: model can memorize train songs, but song-disjoint test accuracy ≈ chance (1/15 ≈ 0.067).
ctrl_path = f"{RES2B}/shuffled_label.json"
if os.path.exists(ctrl_path):
    print('done already:', json.load(open(ctrl_path)))
else:
    rng = np.random.default_rng(7)
    srcs = idxdf.drop_duplicates('source')['source'].values
    fake = dict(zip(srcs, rng.integers(0, N_CLASSES, len(srcs))))
    FAKE_ARR = idxdf['source'].map(fake).values.astype(np.int64)
    tr, te = build_indices('R2', 0)
    if SMOKE: tr, te = tr[:3000], te[:1000]
    tr, va = carve_val_by(tr, 'source', 0)
    _, logits, _ = train_generic('resnet18', tr, va, te, FAKE_ARR, N_CLASSES,
                                 epochs=2 if SMOKE else 15, seed=0)
    m = metric_pack(FAKE_ARR[te], logits.argmax(1), 'seg')
    res = dict(seg_acc=m['seg_acc'], seg_macro_f1=m['seg_macro_f1'], chance=1.0/N_CLASSES)
    json.dump(res, open(ctrl_path, 'w')); print('\nshuffled-label control:', res)
    if SMOKE:
        print('(smoke mode: statistics too small — judge PASS/REVIEW only in the full run)')
    else:
        print('PASS' if abs(res['seg_acc'] - 1.0/N_CLASSES) < 0.05 else
              'REVIEW — accuracy far from chance suggests a pipeline leak')

In [ ]:
# ============ CELL 7 (B5): Artist→genre ORACLE (resumable per fold) ============
# Train a PURE 126-way singer classifier on R2 folds (song-disjoint), then map its predicted artist
# through the deterministic artist->majority-genre table. If oracle genre accuracy ≈ direct R2 genre
# accuracy, the direct model's success is explained by singer identification alone.
ORACLE_FOLDS = range(1) if SMOKE else range(5)
for fold in ORACLE_FOLDS:
    jp = f"{RES2B}/oracle_fold{fold}.json"
    if os.path.exists(jp): print(f"[skip] oracle fold{fold}"); continue
    tr, te = build_indices('R2', fold)
    if SMOKE: tr, te = tr[:3000], te[:1000]
    tr, va = carve_val_by(tr, 'source', fold)
    print(f"[oracle fold{fold}] train={len(tr):,} test={len(te):,} ({N_ARTISTS}-way singer ID)", flush=True)
    _, logits, _ = train_generic('resnet18', tr, va, te, ARTIST_ARR, N_ARTISTS,
                                 epochs=2 if SMOKE else 25, seed=fold)
    part = logits.argmax(1)
    art_m = metric_pack(ARTIST_ARR[te], part, 'artist')                      # raw singer-ID accuracy
    pgen = ART_IDX_2_GENRE[part]                                             # route through the table
    gen_m = metric_pack(GENRE_ARR[te], pgen, 'oracle_genre')
    ys_true, _ = song_level(te, logits, ARTIST_ARR)
    agg = pd.DataFrame(logits); agg['source'] = idxdf.loc[te, 'source'].values
    ps_art = agg.groupby('source').mean().values.argmax(1)
    ys_g = pd.Series(GENRE_ARR[te], index=idxdf.loc[te,'source'].values).groupby(level=0).first().values
    song_gen = metric_pack(ys_g, ART_IDX_2_GENRE[ps_art], 'oracle_genre_song')
    direct = json.load(open(f"{RES2A}/R2_resnet18_fold{fold}.json"))
    out = dict(fold=fold, **art_m, **gen_m, **song_gen,
               direct_seg_acc=direct['seg_acc'], direct_song_acc=direct['song_acc'])
    json.dump(out, open(jp, 'w'))
    print(f"    singer-ID acc={art_m['artist_acc']:.4f} | oracle genre seg={gen_m['oracle_genre_acc']:.4f} "
          f"(direct {direct['seg_acc']:.4f}) | oracle genre song={song_gen['oracle_genre_song_acc']:.4f} "
          f"(direct {direct['song_acc']:.4f})", flush=True)

O = pd.DataFrame([json.load(open(p)) for p in sorted(glob.glob(f"{RES2B}/oracle_fold*.json"))])
if len(O):
    print('\n================ ORACLE SUMMARY ================')
    print(O[['fold','artist_acc','oracle_genre_acc','direct_seg_acc',
             'oracle_genre_song_acc','direct_song_acc']].round(4).to_string(index=False))
    print(f"\nmeans: singer-ID={O.artist_acc.mean():.4f} | oracle genre={O.oracle_genre_acc.mean():.4f} "
          f"vs direct genre={O.direct_seg_acc.mean():.4f}")
    O.to_csv(f"{RES2B}/oracle_summary.csv", index=False)

In [ ]:
# ============ CELL 8 (B6): Artist-count scatter + confusion matrices ============
from scipy.stats import spearmanr

# --- per-genre R3 song accuracy (both models) vs artists-per-genre ---
recs = idxdf.drop_duplicates('source')
apg = recs.groupby('genre_id')['artist_id'].nunique()
rows = []
for mname in ['resnet18', 'crnn']:
    per = {}
    for p in sorted(glob.glob(f"{RES2A}/R3_{mname}_fold*_preds.npz")):
        z = np.load(p, allow_pickle=True)
        ys, ps = song_level(z['gidx'], z['logits'])
        for g in np.unique(ys): per.setdefault(int(g), []).append(float((ps[ys == g] == g).mean()))
    for g, accs in per.items():
        rows.append(dict(model=mname, genre_id=g, genre=ID2GENRE[g],
                         artists=int(apg[g]), r3_song_acc=float(np.mean(accs))))
S = pd.DataFrame(rows)
plt.figure(figsize=(7.5, 5))
for mname, c in zip(['resnet18', 'crnn'], ['#d62728', '#1f77b4']):
    sub = S[S.model == mname]
    rho, pv = spearmanr(sub['artists'], sub['r3_song_acc'])
    plt.scatter(sub['artists'], sub['r3_song_acc'], label=f"{mname} (Spearman ρ={rho:.2f}, p={pv:.3f})",
                color=c, alpha=.8)
for _, r in S[S.model == 'resnet18'].iterrows():
    plt.annotate(r['genre'], (r['artists'], r['r3_song_acc']), fontsize=7,
                 xytext=(3, 3), textcoords='offset points')
plt.xlabel('artists per genre'); plt.ylabel('R3/R4 song-level accuracy')
plt.title('More singers per genre ⇒ less shortcut ⇒ better generalization')
plt.legend(); plt.grid(alpha=.3); plt.tight_layout()
plt.savefig(f"{RES2B}/fig_artists_vs_acc.png", dpi=150); plt.show()
S.to_csv(f"{RES2B}/artists_vs_acc.csv", index=False)

# --- song-level confusion matrices, summed over folds ---
from sklearn.metrics import confusion_matrix
for regime in ['R2', 'R3']:
    YT, YP = [], []
    for p in sorted(glob.glob(f"{RES2A}/{regime}_resnet18_fold*_preds.npz")):
        z = np.load(p, allow_pickle=True)
        ys, ps = song_level(z['gidx'], z['logits']); YT += list(ys); YP += list(ps)
    cm = confusion_matrix(YT, YP, labels=range(N_CLASSES)).astype(float)
    cm = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    plt.figure(figsize=(9, 7.5))
    plt.imshow(cm, cmap='viridis', vmin=0, vmax=1); plt.colorbar(shrink=.8)
    names = [ID2GENRE[i] for i in range(N_CLASSES)]
    plt.xticks(range(N_CLASSES), names, rotation=90, fontsize=8)
    plt.yticks(range(N_CLASSES), names, fontsize=8)
    for i in range(N_CLASSES):
        for j in range(N_CLASSES):
            if cm[i, j] >= 0.15:
                plt.text(j, i, f"{cm[i,j]:.2f}", ha='center', va='center',
                         color='white' if cm[i, j] < .6 else 'black', fontsize=6)
    plt.title(f'{regime} song-level confusion (ResNet-18, row-normalized, all folds)')
    plt.xlabel('predicted'); plt.ylabel('true'); plt.tight_layout()
    plt.savefig(f"{RES2B}/fig_confusion_{regime}.png", dpi=150); plt.show()
print('All 2B artifacts saved to', RES2B)

## After running, send me
1. **B1 probe summary** — especially artist (126-way) vs genre (15-way) means, and the domain probe.
2. **B2 perturbation summary** — the shuffle drop and the low/mid/high mask drops.
3. **B3 MFCC+RF table.**
4. **B4 shuffled-label** PASS/REVIEW line.
5. **B5 oracle summary** — singer-ID accuracy and oracle-vs-direct genre accuracy.
6. **B6** — the Spearman ρ, and the R3 confusion matrix (I want to see where Bauls goes).

That's the complete evidence chapter. Next after this: **2C — WLPA (the three within-language pairs)
and the four playback-singer case study**, which reuses the same checkpoints.
